#02 - Começando Micro-Batch da Bronze

Lê as atualizações a cada 1 minuto de cada ação e salva na tabela bronze dos micro batch
---


---

Instalando yFinance para consulta e atualização de preços

---

In [0]:
%pip install yfinance 
dbutils.library.restartPython() 

---

Importando todas as bibliotecas que serão utilizadas durante o notebook, em seguida definimos todas as váriaveis que serão utilizadas durante a execução.

# .
### Variáveis e suas utilizações
bronze_path = define caminho que será criado a tabela no Unity Catalog.

tickers = informa os tickers que serão buscados.

pause_batch = informa o tempo em segundos para recomeçar a busca das novas atualizações das ações.

periodo_batch = informa qual o periodo de busca em relação a data atual.

intervalo_batch = informa o intervalo de tempo de cada variação da ação.

---

In [0]:
import yfinance as yf
import pyspark.sql.functions as sf
import pandas as pd
import time
import pytz

bronze_path = "workspace.stocks.micro_batch_bronze"
tickers = ["PETR4.SA", "VALE3.SA", "BBAS3.SA"]

#-----Utilizado durante o polling de dados----- 
pause_batch = 60
periodo_batch = "1d"
intervalo_batch = "1m"

---

# Execução do Micro Batch em Loop
### Dentro do loop:

Começamos procurando a última data e horario que foi feito o salvamento na tabela delta, caso não exista retorna vazio.

É feito o download das informações do dia inteiro para cada ação dentro array de ticks.

Conforme formato que retorna os dados do yfinance, resetamos os index para o datetime voltar como coluna e não index, achatamos os MultiIndex das colunas, colocamos os nomes de todas as colunas da tabela em minúsculo para padronizar os nomes.

Posteriormente retiramos o timezone da coluna datetime, para evitar conflitos de fuso horário na comparação e conversão posterior.

Verificamos se existe um último TimeStamp para realizar a filtragem dos dados antes de salvar, evitando uma duplicidade de informações na tabela delta.

Salvamos em um array cada dataframe gerado pelo ciclo do for loop caso existam registros novos desde o último salvamento. Concatenamos esses três dataframes separados dentro de um só com o pd.concat, o novo dataframe concatenado convertemos para um dataframe spark.

Após converter a coluna datetime para o formato datetime, escrevemos/salvamos ela dentro da tabela delta de micro-batch particionada por ticker.

---

In [0]:
while True:
    tickers_batch = []
    for t in tickers:
        print(f"-----Micro Batch de {t}-----")
        try:
            ultimo_ts = (
                spark.read.table(bronze_path)\
                .filter(sf.col("ticker") == t)\
                .agg(sf.max("datetime"))\
                .collect()[0][0]
            )
            print(f"-Ultimo TimeStamp para {t}: {ultimo_ts}-")
        except:
            ultimo_ts = None
    
        try:
            stock = yf.download(t, 
                                period=periodo_batch, 
                                interval=intervalo_batch,
                                progress=False)

            stock = stock.reset_index()
            stock.columns = stock.columns.get_level_values(0)

            stock.columns = [c.lower() for c in stock.columns]
            stock['datetime'] = stock['datetime'].dt.tz_localize(None)
            stock['ticker'] = t
            stock['fonte'] = 'micro_batch'

            if ultimo_ts is not None:
                stock = stock[stock['datetime'] > ultimo_ts]

            if stock.empty:
                print(f"-----Micro Batch de {t} sem atualizações-----")
                print("--------------------")
            else:
                print(f"-----Micro Batch de {t} encontrado-----")
                print("--------------------")
                tickers_batch.append(stock)

        except Exception as e:
            print(f"-----Micro Batch de {t} não encontrado-----")
            print(f"Verifique o erro: {e}")
            print("--------------------")
    
    if tickers_batch:
        tickers_batch_df = pd.concat(tickers_batch, ignore_index=True)

        df_batch = spark.createDataFrame(tickers_batch_df)
        df_batch = df_batch.withColumn("datetime",sf.to_timestamp(sf.col("datetime")))
        
        (df_batch.write
         .format("delta")
         .mode("append")
         .partitionBy("ticker")
         .saveAsTable(bronze_path))
        
        print(f"{len(tickers_batch_df)} registros salvos na Bronze")
        print("--------------------")
    time.sleep(pause_batch)
